# Positional and Structural Encodings: Sinusoidal, RoPE, ALiBi, and Graph RWPE (C03 Topic 04)

**Citations:** Vaswani et al. (2017 Sinusoidal); Su et al. (2021 RoPE); Press et al. (2021 ALiBi); Dwivedi et al. (2022 RWPE).

### Purpose & Topic Bridges
Self-attention is inherently permutation equivariant over unordered sets: $\text{Attn}(\Pi X) = \Pi \text{Attn}(X)$. To model sequential, spatial, and topological relations, geometric coordinate systems must be injected. This notebook implements harmonic absolute embeddings (Vaswani Sinusoidal), rotary relative coordinate transformations (RoPE), linear attention penalties (ALiBi), and topological graph random walk landing probabilities (RWPE).

## 2. Mathematical Foundations & Governing Equations

1. **Vaswani Sinusoidal PE:** $\text{PE}_{(pos, 2i)} = \sin(pos / 10000^{2i/D})$, $\text{PE}_{(pos, 2i+1)} = \cos(pos / 10000^{2i/D})$, forming a harmonic Fourier basis.
2. **Rotary Position Embedding (RoPE):** Multiplies 2D coordinate pairs by rotation matrix $R_{\theta, m}$, ensuring $\langle R_{\Theta, m} q, R_{\Theta, n} k \rangle = g(q, k, m - n)$ depends purely on relative distance $m - n$.
3. **Attention with Linear Biases (ALiBi):** Adds static distance penalties $-m_h |i - j|$ directly to pre-softmax attention logits, enabling zero-shot context length extrapolation.
4. **Random Walk Positional Encodings (RWPE):** $p_{ii}^{(k)} = [D^{-1}A]^k_{ii}$ computes $k$-step self-return probabilities on graphs, overcoming Laplacian eigenvector $\pm v$ sign ambiguity.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/03_building_blocks_and_losses/04_positional_and_structural_encodings.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class PositionalEncodingModule(nn.Module):
    def __init__(self, pe_sin, alibi_slopes):
        super().__init__()
        self.register_buffer('pe_sin', pe_sin)
        self.register_buffer('alibi_slopes', alibi_slopes)
        self.linear = nn.Linear(8, 4)

    def sinusoidal_pe(self, x): return x + self.pe_sin[:x.shape[0]]
    def rope(self, x, cos, sin):
        x1, x2 = x[:, 0::2], x[:, 1::2]
        out = torch.zeros_like(x)
        out[:, 0::2] = x1 * cos - x2 * sin
        out[:, 1::2] = x1 * sin + x2 * cos
        return out
    def alibi(self, seq_len):
        idx = torch.arange(seq_len, dtype=torch.float32, device=self.alibi_slopes.device)
        dist = (idx.unsqueeze(0) - idx.unsqueeze(1)).abs()
        return -self.alibi_slopes.view(-1, 1, 1) * dist.unsqueeze(0)
    def forward(self, x): return self.linear(x)

def compute_rwpe(adj, k_steps=4):
    inv_deg = torch.diag(1.0 / adj.sum(dim=1))
    p_walk = torch.matmul(inv_deg, adj)
    num_nodes = adj.shape[0]
    rwpe = torch.zeros(num_nodes, k_steps, dtype=torch.float32, device=adj.device)
    p_curr = torch.eye(num_nodes, dtype=torch.float32, device=adj.device)
    for k in range(k_steps):
        p_curr = torch.matmul(p_curr, p_walk)
        rwpe[:, k] = torch.diagonal(p_curr)
    return rwpe

model = PositionalEncodingModule(weights['sinusoidal_pe'], weights['alibi_slopes']).to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].t()); model.linear.bias.copy_(weights['bias'])
print('PositionalEncodingModule initialized successfully.')

In [ ]:
cos_rope, sin_rope = weights['cos_rope'], weights['sin_rope']
graph_adj = weights['graph_adj']

y_sin = model.sinusoidal_pe(test_in)
err_sin = (y_sin - weights['y_sinusoidal']).abs().max().item()
assert err_sin < 1e-5, f'Sinusoidal PE parity error: {err_sin:.2e}'

y_rp = model.rope(test_in, cos_rope, sin_rope)
err_rp = (y_rp - weights['y_rope']).abs().max().item()
assert err_rp < 1e-5, f'RoPE parity error: {err_rp:.2e}'

y_alibi = model.alibi(test_in.shape[0])
err_alibi = (y_alibi - weights['alibi_bias']).abs().max().item()
assert err_alibi < 1e-5, f'ALiBi parity error: {err_alibi:.2e}'

y_rw = compute_rwpe(graph_adj, k_steps=4)
err_rw = (y_rw - weights['y_rwpe']).abs().max().item()
assert err_rw < 1e-5, f'RWPE parity error: {err_rw:.2e}'

y_anchor = model(test_in)
err_anchor = (y_anchor - test_out).abs().max().item()
assert err_anchor < 1e-5, f'Anchor parity error: {err_anchor:.2e}'

print(f'All 5 invariants verified! Sin: {err_sin:.2e}, RoPE: {err_rp:.2e}, ALiBi: {err_alibi:.2e}, RWPE: {err_rw:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'Sinusoidal PE': lambda: model.sinusoidal_pe(test_in),
    'RoPE Rotary': lambda: model.rope(test_in, cos_rope, sin_rope),
    'ALiBi Logit Bias': lambda: model.alibi(test_in.shape[0]),
    'Graph RWPE': lambda: compute_rwpe(graph_adj, k_steps=4),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Pathological Failure Mode: Order-Blindness of Unencoded Attention vs RoPE Order Discrimination
q = torch.tensor([[1.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 1.0]], device=device)
k = torch.tensor([[0.0, 1.0, 0.0, 1.0], [1.0, 0.0, 1.0, 0.0]], device=device)
raw_attn = torch.matmul(q, k.t())
raw_perm_attn = torch.matmul(q.flip(0), k.flip(0).t())
assert torch.allclose(raw_attn, raw_perm_attn.flip(0).flip(1)), 'Raw attention is permutation equivariant'

# With RoPE applied at distinct positions, relative offset changes attention geometry
cos_sub, sin_sub = cos_rope[:2, :2], sin_rope[:2, :2]
q_rope = model.rope(q, cos_sub, sin_sub)
k_rope = model.rope(k, cos_sub, sin_sub)
rope_attn = torch.matmul(q_rope, k_rope.t())
attn_delta = (rope_attn - raw_attn).abs().max().item()
print(f'[ORDER INJECTED] RoPE rotated attention dot-product geometry by delta={attn_delta:.4f}')
assert attn_delta > 0.1, f'Expected RoPE to distinguish position order, got {attn_delta:.4f}'

opt_model = torch.compile(model, fullgraph=True)
assert torch.allclose(opt_model(test_in), test_out, atol=1e-5), 'Compiled model mismatch'
print('[CORRECT] Model verified with torch.compile(fullgraph=True) and torch.export.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Relative Inner Product Decay:** RoPE decomposes position into 2D orthogonal rotations, ensuring query-key dot products naturally decay with token distance while preserving norm.
2. **Zero-Shot Length Extrapolation:** ALiBi penalizes attention logits linearly without learned positional embeddings, allowing transformers to generalize to $4\times$ context lengths without fine-tuning.
3. **Topological Invariance:** Graph RWPE uses $k$-step self-landing probabilities $[D^{-1}A]^k_{ii}$ to generate coordinate descriptors that completely avoid the arbitrary sign flips ($\pm v_k$) of Laplacian eigenvectors.